In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
import os
from langchain_community.document_loaders import TextLoader

docs = []

repo_path = r"D:\Langchain\Prac\kthena"
extensions = (
    ".py",
    ".md",     
    ".txt",
    ".json",
    ".yaml",
    ".yml"
)
for root, dirs, files in os.walk(repo_path):
    for file in files:
        if file.endswith(extensions):
            try:
                loader = TextLoader(os.path.join(root, file), encoding="utf-8")
                docs.extend(loader.load())
            except Exception as e:
                print(f"Error loading {file}: {e}")

print(f"Loaded {len(docs)} files")

Loaded 665 files


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents = splitter.split_documents(docs)

print(len(documents))

6412


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_10116\2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(
    documents,
    embeddings
)

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0.5
)

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

In [8]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.

<context>
{context}
</context>

Question: {input}
""")

In [10]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [15]:
response = retrieval_chain.invoke({
    "input": "What Kubernetes components are used?"
})
print(response["input"])
print(response["answer"])

What Kubernetes components are used?
Based on the provided context, the following Kubernetes components are used:

*   **CRD (Custom Resource Definition)**
*   **Controller** (described as a "Kubernetes-native component")
*   **Pods**
*   **Custom Resources (CRs)**
